### Exploring sythetic prompt generation as prompt protection

In this notebook, you'll work on finding ways to create synthetic prompts that mirror original prompts. You'll use a few different methods to do so and see both some potential use cases but also where this strategy might fail. 

Before running this notebook, you'll need to install a [llamafile](https://github.com/Mozilla-Ocho/llamafile) and start it in the background by:

1. Making the file executable
2. Running this executable

It should open a browser. We'll use it from here though!

For the later part of the notebook, you'll either use [Hugging Face models](https://huggingface.co/) (if you have an account and also a computer that can run them) or [ollama](https://ollama.com/). For ollama, you can also [install the software](https://ollama.com/) if you'd like to do more local-first AI, but it should also Just Work with the pip library.

In [ ]:
from openai import OpenAI
import ollama
import torch
from time import time
from datetime import timedelta

### Initial prompt engineering

Our goal is to see just how far we can get from prompt engineering alone. Can prompt engineering help us sanitize and pseudonymize prompts? Let's try it out with our llamafile and writing some clever prompts. You can reuse any of the prompts you used in the Presidio notebook or take a look at the "Master Class - Generating Sensitive Prompts" notebook or data/sensitive_prompts.json 

First, let's start with something that mirrors the Presidio task.


In [ ]:
llamafile_client = OpenAI(
    base_url="http://localhost:8080/v1", # "http://<Your api-server IP>:port"
    api_key = "sk-no-key-required"
)

In [ ]:
initial_prompt = """
Take the text below.
Locate names, addresses, URLs, phone numbers, email addresses, identity numbers, dates, credit card numbers, bank account numbers, cryptographic keys or usernames. 
Replace these sensitive tokens with a REDACTED token. 

Here is the text to fix:

{text}
"""

In [ ]:
sensitive_text = "His name is Mr. Samuel Jones and his phone number is +49 110 0342121345"

In [ ]:
initial_prompt.format(text=sensitive_text)

In [ ]:
first_output = llamafile_client.chat.completions.create(
    model="LLaMA_CPP",
    messages=[{'role': 'user', 'content': initial_prompt.format(text=sensitive_text)}]
)

print(first_output)

In [ ]:
first_output.choices[0].message.content

Thanks, I think... lol. 😂

Give it a try with a few other texts or modify the prompt, does it find the right tokens? 

# Leveraging Few-shot / In-Context Learning

Few-shot and in-context learning is a way to on-the-fly generate different responses or to recontextualize the in-memory information to better perform a task. Often this is used to complete simple language-tasks or other tasks by showing examples, but it can also be used to write better prompts.

To invoke in-context, usually multiple examples are shown. For example:

```
Rewrite texts to remove potentially sensitive tokens.
Locate names, addresses, URLs, phone numbers, email addresses, identity numbers, dates, credit card numbers, bank account numbers, cryptographic keys or usernames. 
Replace these sensitive tokens with a REDACTED token. 

Follow these examples:

His name is Mr. Samuel Jones and his phone number is +49 110 0342121345 // His name is [REDACTED] and his phone number is [REDACTED]
Sure, my drivers license is B4382947-8. Its tied to my same last name Yarok. // Sure, my drivers license is [REDACTED]. Its tied to my same last name [REDACTED].
Janet arrived at the emergency room yesterday evening at 11:24pm and was immediately taken into the ICU. Her vitals are stable and her daughter Jessica stayed overnight to keep an eye on her. If you want to reach the nurses line directly call 777-1138. // [REDACTED] arrived at the emergency room [REDACTED] and was immediately taken into the ICU. Her vitals are stable and [REDACTED] stayed overnight to keep an eye on her. If you want to reach the nurses line directly call [REDACTED]. 
{text} //
```

In [ ]:
in_context_prompt = """
Rewrite texts to remove potentially sensitive tokens.
Locate names, addresses, URLs, phone numbers, email addresses, identity numbers, dates, credit card numbers, bank account numbers, cryptographic keys or usernames. 
Replace these sensitive tokens with a REDACTED token. 

Follow these examples:

His name is Mr. Samuel Jones and his phone number is +49 110 0342121345 // His name is [REDACTED] and his phone number is [REDACTED]
Sure, my drivers license is B4382947-8. Its tied to my same last name Yarok. // Sure, my drivers license is [REDACTED]. Its tied to my same last name [REDACTED].
Janet arrived at the emergency room yesterday evening at 11:24pm and was immediately taken into the ICU. Her vitals are stable and her daughter Jessica stayed overnight to keep an eye on her. If you want to reach the nurses line directly call 777-1138. // [REDACTED] arrived at the emergency room [REDACTED] and was immediately taken into the ICU. Her vitals are stable and [REDACTED] stayed overnight to keep an eye on her. If you want to reach the nurses line directly call [REDACTED]. 
{text} //
"""

In [ ]:
more_text = """
Yo, can you please call me back -- you have my number under JJ already. It's urgent because your dad is in the hospital and they are asking for his identity details. I htink his SS number starts with 444-22- but I forget if the last numbers are 5543 or 5544. Call me back !"""

In [ ]:
in_context_prompt.format(text=more_text)

In [ ]:
second_output = llamafile_client.chat.completions.create(
    model="LLaMA_CPP",
    messages=[{'user': in_context_prompt.format(text=more_text)}]
)

print(second_output)

Uhhhhh... now you try with a few additional in-context examples AND some more tricky text!

## Crescendo attack as inspiration

Microsoft released [a paper and adversarial approach](https://crescendo-the-multiturn-jailbreak.github.io/) for jailbreaking LLMs called Crescendo. In this approach, the attacker tries to generate increasingly adversarial prompts pointing the entire chat history/conversation slowly into what would before be banned content (i.e. illegal, explicit, harrassment/hate speech.)

To automate this, the attack utilizes more than one LLM at once to steer the conversation towards some goal. 

The idea is:

1. Setup one model as the model you want to attack. You can do that via HuggingFace/llamafile or via a model running on the LAN. 
2. Setup one model as the model that will help you attack the other model to produce increasingly more feminist responses. You can do that via HuggingFace/llamafile or via a model running on the LAN.
3. (optional): if you need a model to review potential guardrails, you might want to try getting [LlamaGuard](https://github.com/meta-llama/PurpleLlama) running. Note that this will require 10s of gigs of memory. (I know :()
4. Start manually and guide one conversation to the next. One tip from the research is to have the helper model summarize the other model's response and then nudge toward the goal.
5. If it works, you can try to see if there is an automatable attack template. Check out the research for the pseudocode/example.


We are going to use it to try to better sanitize our prompts. Let's go!

In [ ]:
ollama_client = ollama.Client()
model_name = 'gemma3:4b'

In [ ]:
crescendo_init_prompt = """
Rewrite texts to remove potentially sensitive tokens.
Locate names, addresses, URLs, phone numbers, email addresses, identity numbers, dates, credit card numbers, bank account numbers, cryptographic keys or usernames. 
Locate partial identifiers, like partial ID numbers or things like family relationships.
Replace these sensitive tokens with a REDACTED token. 

Follow these examples:

His name is Mr. Samuel Jones and his phone number is +49 110 0342121345 // His name is [REDACTED] and his phone number is [REDACTED]
Sure, my drivers license is B4382947-8. Its tied to my same last name Yarok. // Sure, my drivers license is [REDACTED]. Its tied to my same last name [REDACTED].
Janet arrived at the emergency room yesterday evening at 11:24pm and was immediately taken into the ICU. Her vitals are stable and her daughter Jessica stayed overnight to keep an eye on her. If you want to reach the nurses line directly call 777-1138. // [REDACTED] arrived at the emergency room [REDACTED] and was immediately taken into the ICU. Her vitals are stable and [REDACTED] stayed overnight to keep an eye on her. If you want to reach the nurses line directly call [REDACTED]. 
{text} //
"""


In [ ]:
cres_first_output = llamafile_client.chat.completions.create(
    model="LLaMA_CPP",
    messages=[{'role': 'user', 'content': crescendo_init_prompt.format(text=sensitive_text)}]
)

print(cres_first_output)

In [ ]:
sensitive_text

In [ ]:
cres_discriminator_prompt = """

Your job is to find errors in sensitive token redaction. Sensitive tokens include things like:

- names
- phone numbers
- identity numbers
- partial identity numbers
- credit card numbers 
- partial names
- family relationships
- dates
- credit card numbers
- bank account numbers
- cryptographic keys
- usernames

This data should be replaced by a [REDACTED] token.

Review the following text: 

{text}

Summarize the redaction errors and give a better prompt to guide how to redact sensitive information based on the errors seen. 
"""


In [ ]:
messages = [
    {'role': 'user',
     'content': crescendo_init_prompt
    }, 
    {
     'role': 'agent',
     'content': cres_first_output.choices[0].message.content,
    },
    {'role': 'user',
     'content': cres_discriminator_prompt.format(text=cres_first_output.choices[0].message.content),
    }
]
     

In [ ]:
cres_disc_response = ollama_client.chat(
    model=model_name,
    messages=messages,
)
print(cres_disc_response.message.content)

In [ ]:
disc_prompt = cres_disc_response.message.content.split('**Revised Prompt:**')[1].split('---')[0]

In [ ]:
disc_prompt

In [ ]:
full_prompt = disc_prompt + 'Please redact this text following the above instructions: {text}'

In [ ]:
cres_third_output = llamafile_client.chat.completions.create(
    model="LLaMA_CPP",
    messages=[{'role': 'user', 'content': full_prompt.format(text=sensitive_text)}]
)

In [ ]:
cres_third_output

Okay! Now we might want to play with parsable rewriting....

### Hugging Face Addendum! :) 

Note: the code in the next cell is only if you have a GPU or a M1/M2 silicon.. if not, please just use more llamafiles or a LAN model. Alternatively, you can set up and use an API for a popular LLM.

- If you have one or more GPUs, please change your device below to map to your GPU, sometimes just "cuda" works.
- If you have a Macbook M1 or M2, please make sure you have [pytorch for MPS installed](https://developer.apple.com/metal/pytorch/).

To use the meta-llama repository you must also:

1. Create a HuggingFace account.
2. Install hugging face and cli: `pip install -U "huggingface_hub[cli]"`
3. Request access to the meta-llama3 models, which you can do [on the model page](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct). Once they approve you will get an email.
5. Log into hugging face CLI using your user token (create one in User->Settings). Set the token to Read to just download models. Command is `huggingface-cli login`.
6. Then proceed with code below!

Continue or switch models and way that you are steering. Work with a partner to explore what paths you find interesting!

In [ ]:
# Only run this after reading above note! :)
from transformers import pipeline


pipe = pipeline("text-generation", 
                model="meta-llama/Llama-3.2-1B-Instruct", # you can also choose other huggingface models here!
                torch_dtype=torch.bfloat16, 
                device="mps", 
                max_new_tokens=500) # here is where to change your device if you use something other than mac silicon